# Yogurt brand choice — `torch-choice` reproduces R/`mlogit`

This tutorial fits a multinomial logit model of household yogurt brand choice
on the Jain–Vilcassim–Chintagunta (JBES, 1994) panel. We fit the same model
twice — once in R with `mlogit`, once in Python with `torch-choice` — and show
that the two implementations recover the same coefficients to numerical
precision. We then explore richer alternative specifications and discuss
which is theoretically preferred.


## 1. About this dataset

**Domain.** Consumer packaged goods / brand choice. 100 households make 2,412
yogurt purchases over time, choosing one of four brands per trip:

| Brand | Notes |
|---|---|
| `yoplait` | Reference (premium) |
| `dannon` | Mass-market |
| `hiland` | Local (regional) |
| `weight` | "Weight Watchers" |

Each row is one purchase occasion. For every occasion we observe the **price**
charged for all four brands and a **feature** indicator (`feat.*` = 1 if the
brand was on a newspaper feature/promotion that week). The chosen brand is in
the `choice` column.

**Reference.** Jain, D. C., Vilcassim, N. J., Chintagunta, P. K. (1994).
"A Random-Coefficients Logit Brand-Choice Model Applied to Panel Data."
*Journal of Business & Economic Statistics* 12(3), 317–328.
([doi:10.1080/07350015.1994.10524547](https://doi.org/10.1080/07350015.1994.10524547))

**License.** Distributed in the R package `Ecdat` under GPL-2, redistributable
with citation.

### How to download

The CSV in this folder was extracted from the cran/Ecdat GitHub mirror:

```python
import pyreadr
rda = pyreadr.read_r_url(
    "https://raw.githubusercontent.com/cran/Ecdat/master/data/Yogurt.rda"
)
yogurt = list(rda.values())[0]   # only one DataFrame in the .rda
```

Or, in R:

```r
data(Yogurt, package = "Ecdat")
write.csv(Yogurt, "yogurt.csv", row.names = FALSE)
```

The notebook reads `yogurt.csv` from this folder.


In [1]:
# Make `tutorials/_mlogit_compare.py` importable from this folder.
import math
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from torch_choice.data import ChoiceDataset, utils
from torch_choice.model import ConditionalLogitModel

from _mlogit_compare import run_or_load_mlogit, compare_coefs

torch.manual_seed(0)   # reproducibility across re-runs (LBFGS line search)
HERE = Path.cwd()


In [2]:
df = pd.read_csv(HERE / "yogurt.csv")
print(f"shape={df.shape}, n_households={df['id'].nunique()}, n_occasions={len(df)}")
df.head()


shape=(2412, 10), n_households=100, n_occasions=2412


,id,feat.yoplait,feat.dannon,feat.hiland,feat.weight,price.yoplait,price.dannon,price.hiland,price.weight,choice
0,1.0,0.0,0.0,0.0,0.0,10.8,8.1,6.1,7.9,weight
1,1.0,0.0,0.0,0.0,0.0,10.8,9.8,6.4,7.5,dannon
2,1.0,0.0,0.0,0.0,0.0,10.8,9.8,6.1,8.6,dannon
3,1.0,0.0,0.0,0.0,0.0,10.8,9.8,6.1,8.6,dannon
4,1.0,0.0,0.0,0.0,0.0,12.5,9.8,4.9,7.9,dannon


## 2. R/`mlogit` reference fit

We fit a textbook MNL on the wide-format CSV using `mlogit::mlogit`, with
`yoplait` as the reference alternative:

```r
suppressPackageStartupMessages(library(mlogit))
df   <- read.csv("yogurt.csv")
data <- dfidx(df, shape="wide", choice="choice", varying=2:9,
              sep=".", idnames=c("chid","alt"))
mod  <- mlogit(choice ~ price + feat, data=data, reflevel="yoplait")
summary(mod)
```

The full R script is in `fit_mlogit.R` next to this notebook. The cell below
calls it via `Rscript`; if R isn't installed it transparently falls back to the
cached output in `mlogit_output.json` (also next to this notebook), so the
comparison still renders for readers without an R installation.


In [3]:
mlogit_df, mlogit_ll = run_or_load_mlogit(
    r_script_path=HERE / "fit_mlogit.R",
    csv_path=HERE / "yogurt.csv",
    cache_path=HERE / "mlogit_output.json",
)
print(f"R/mlogit train log-likelihood: {mlogit_ll:.4f}")
mlogit_df


[mlogit] Live R fit succeeded (fit_mlogit.R).
R/mlogit train log-likelihood: -2656.8879


,name,estimate,std_err,z_value,p_value
0,(Intercept):dannon,-0.734571,0.080644,-9.108791,0.000000
1,(Intercept):hiland,-4.450166,0.187118,-23.782711,0.000000
2,(Intercept):weight,-1.375755,0.088982,-15.461097,0.000000
3,price,-0.366584,0.024366,-15.044876,0.000000
4,feat,0.491433,0.120063,4.093129,0.000043


## 3. `torch-choice` fit

The same MNL specification expressed in `torch-choice`'s formula syntax is

```
(itemsession_price|constant) + (itemsession_feat|constant) + (intercept|item)
```

— one shared coefficient on `price`, one shared coefficient on `feat`, and an
alt-specific intercept (the first item, `yoplait`, is the reference, with
intercept pinned to zero, which matches R's `reflevel="yoplait"`).

### Build a `ChoiceDataset`

The CSV is in *wide* format (one row per occasion, with brand-suffixed price /
feat columns). We reshape to long format and pivot into the 3D
`(num_sessions, num_items, num_features)` tensors `torch-choice` expects.


In [4]:
# Brand encoding: index 0 is the reference (matches R's reflevel="yoplait").
BRAND_ORDER = ["yoplait", "dannon", "hiland", "weight"]
brand_to_idx = {b: i for i, b in enumerate(BRAND_ORDER)}

# Wide -> long: 2,412 occasions x 4 brands = 9,648 rows.
df = df.copy()
df["occasion"] = np.arange(len(df))
long_records = []
for brand, idx in brand_to_idx.items():
    long_records.append(
        df[["occasion", f"price.{brand}", f"feat.{brand}"]]
        .rename(columns={f"price.{brand}": "price", f"feat.{brand}": "feat"})
        .assign(brand=idx)
    )
long_df = pd.concat(long_records, ignore_index=True).sort_values(["occasion", "brand"])

# Pivot into (num_sessions, num_items, num_features) tensors.
itemsession_price = utils.pivot3d(long_df, dim0="occasion", dim1="brand", values="price")
itemsession_feat  = utils.pivot3d(long_df, dim0="occasion", dim1="brand", values="feat")

# Per-occasion chosen brand index, household id, and session id.
item_index    = torch.LongTensor(df["choice"].map(brand_to_idx).values)
user_index    = torch.LongTensor(df["id"].astype(int).values - 1)  # 0-indexed
session_index = torch.arange(len(df))
NUM_USERS     = int(user_index.max().item()) + 1   # 100 households
N_OCC         = len(item_index)                    # 2412 occasions

dataset = ChoiceDataset(
    item_index=item_index,
    num_items=4,
    num_users=NUM_USERS,
    user_index=user_index,
    session_index=session_index,
    itemsession_price=itemsession_price,
    itemsession_feat=itemsession_feat,
)
print(dataset)


ChoiceDataset(num_items=4, num_users=100, num_sessions=2412, label=[], item_index=[2412], user_index=[2412], session_index=[2412], item_availability=[], itemsession_price=[2412, 4, 1], itemsession_feat=[2412, 4, 1], device=cpu)


/Users/tianyudu/Development/torch-choice/torch_choice/data/utils.py:28: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  tensor_slice.append(torch.Tensor(layer[dim1_list].values))
/Users/tianyudu/Development/torch-choice/torch_choice/data/choice_dataset.py:286: UserWarning: The number of sessions is inferred from the number of unique sessions in the session_index tensor. This might lead to unexpected behaviors if some sessions never appeared in the session_index tensor. For a safer behavior, please provide the number of sessions explicitly by using the num_sessions keyword while initia

### Fit (Spec A — pooled MNL)

We use full-batch LBFGS for 1,000 epochs — the same recipe the package's
`paper_demo.py` uses for ModeCanada. `model.fit(...)` returns an
`EstimationOutput` whose `coef_summary` mirrors a standard regression table.


In [5]:
model = ConditionalLogitModel(
    formula="(itemsession_price|constant) + (itemsession_feat|constant) + (intercept|item)",
    dataset=dataset,
    num_items=4,
)
print(model)


ConditionalLogitModel(
  (coef_dict): ModuleDict(
    (itemsession_price[constant]): Coefficient(variation=constant, num_items=4, num_users=None, num_params=1, 1 trainable parameters in total, initialization=normal, device=cpu).
    (itemsession_feat[constant]): Coefficient(variation=constant, num_items=4, num_users=None, num_params=1, 1 trainable parameters in total, initialization=normal, device=cpu).
    (intercept[item]): Coefficient(variation=item, num_items=4, num_users=None, num_params=1, 3 trainable parameters in total, initialization=normal, device=cpu).
  )
)
Conditional logistic discrete choice model, expects input features:

X[itemsession_price[constant]] with 1 parameters, with constant level variation.
X[itemsession_feat[constant]] with 1 parameters, with constant level variation.
X[intercept[item]] with 1 parameters, with item level variation.
device=cpu


In [6]:
result = model.fit(
    dataset,
    batch_size=-1,
    learning_rate=0.01,
    num_epochs=1000,
    model_optimizer="LBFGS",
    backend="lightning",
    print_summary=False,
)
print(result)
print(f"\ntorch-choice train log-likelihood: {result.train_ll:.4f}")


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Starting PyTorch Lightning training loop.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

  | Name  | Type                  | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model | ConditionalLogitModel | 5      | train | 0    
----------------------------------------------------------------
5         Trainable params
0         Non-trainable params
5         Total params
0.000     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode
0         Total Flops


/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1000` reached.


==================== model results ====================
Log-likelihood: [Training] -2656.888, [Validation] None, [Test] None

| Coefficient                   |   Estimation |   Std. Err. |   z-value | Pr(>|z|)   | Significance   |
|:------------------------------|-------------:|------------:|----------:|:-----------|:---------------|
| itemsession_price[constant]_0 |    -0.366582 |   0.0243659 |   -15.045 | < 2e-16    | ***            |
| itemsession_feat[constant]_0  |     0.491423 |   0.120063  |     4.093 | 4.257e-05  | ***            |
| intercept[item]_0             |    -0.734573 |   0.080644  |    -9.109 | < 2e-16    | ***            |
| intercept[item]_1             |    -4.45013  |   0.187116  |   -23.783 | < 2e-16    | ***            |
| intercept[item]_2             |    -1.37575  |   0.0889814 |   -15.461 | < 2e-16    | ***            |
Significance codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

torch-choice train log-likelihood: -2656.8879


## 4. Side-by-side comparison

The two implementations expose coefficient names differently. `mlogit`
labels alternative-specific intercepts as `(Intercept):dannon` etc., while
`torch-choice` indexes them positionally inside `intercept[item]_*`. The
mapping below is the only per-dataset adapter required for the comparison
helper.


In [7]:
NAME_MAP = {
    "itemsession_price[constant]_0": "price",
    "itemsession_feat[constant]_0":  "feat",
    "intercept[item]_0":             "(Intercept):dannon",
    "intercept[item]_1":             "(Intercept):hiland",
    "intercept[item]_2":             "(Intercept):weight",
}
diff = compare_coefs(mlogit_df, result, NAME_MAP)
print(
    f"\nLL: mlogit={mlogit_ll:.4f}  "
    f"torch-choice={result.train_ll:.4f}  "
    f"abs_diff={abs(mlogit_ll - result.train_ll):.2e}"
)
diff


[compare_coefs] estimates: max |diff| = 3.856e-05 (0.0022%); tol 1e-03 -> PASS
[compare_coefs] std errs:  max |diff| = 1.777e-06 (0.0009%); tol 1e-03 -> PASS

LL: mlogit=-2656.8879  torch-choice=-2656.8879  abs_diff=6.15e-05


,coef,mlogit_est,tc_est,est_abs_diff,est_pct_diff,mlogit_se,tc_se,se_abs_diff,se_pct_diff
0,itemsession_price[constant]_0,-0.366584,-0.366582,0.000003,0.000724,0.024366,0.024366,1.176509e-07,0.000483
1,itemsession_feat[constant]_0,0.491433,0.491423,0.000011,0.002172,0.120063,0.120063,2.266068e-07,0.000189
2,intercept[item]_0,-0.734571,-0.734573,0.000001,0.000195,0.080644,0.080644,2.345911e-07,0.000291
3,intercept[item]_1,-4.450166,-4.450128,0.000039,0.000867,0.187118,0.187116,1.777026e-06,0.000950
4,intercept[item]_2,-1.375755,-1.375749,0.000007,0.000495,0.088982,0.088981,3.240716e-07,0.000364


**Conclusion of §4.** `torch-choice` reproduces `mlogit`'s MNL coefficient
estimates and log-likelihood to within float32 round-off. This validates the
package's optimizer and standard-error implementation against an established
reference.


## 5. Alternative specifications and theoretical recommendation

The pooled MNL fit in §3–§4 (call it **Spec A**) imposes a strong assumption:
*every household responds identically to price changes and to newspaper
features*. The original Jain–Vilcassim–Chintagunta (1994) paper questions
exactly this assumption — its central contribution is a **random-coefficients
logit** that lets price- and feature-sensitivity vary across households. They
argue the panel structure (~24 purchases per household on average) makes
heterogeneity identifiable, and they find substantial taste variation: the
estimated standard deviation of the price coefficient is comparable in
magnitude to its mean.

### Specs to compare

| Spec | Formula | What changes | n_params |
|---|---|---|---|
| **A** (pooled MNL — current) | `(itemsession_price\|constant) + (itemsession_feat\|constant) + (intercept\|item)` | one price coef, one feat coef, three brand intercepts | 5 |
| **B** (household price sensitivity) | `(itemsession_price\|user) + (itemsession_feat\|constant) + (intercept\|item)` | each of 100 households gets its own price coefficient; feat shared | 100+1+3 = 104 |
| **C** (full household heterogeneity) | `(itemsession_price\|user) + (itemsession_feat\|user) + (intercept\|item)` | each household gets its own price *and* feat coefficient | 100+100+3 = 203 |

Specs B and C are *fixed-effects* approximations to the JVC 1994
random-coefficients logit. They estimate one coefficient per household
(treating heterogeneity as known finite parameters) instead of integrating
over a continuous mixing distribution. With 100 households and ~24 purchases
each, the FE approach is feasible; with thousands of users it would not be.

### Fit Specs B and C


In [8]:
def fit_spec(formula: str, label: str, *, num_epochs: int = 1000,
             optimizer: str = "LBFGS") -> tuple[float, int, "EstimationOutput"]:
    """Fit a torch-choice spec; return (train_ll, n_params, result)."""
    torch.manual_seed(0)
    m = ConditionalLogitModel(
        formula=formula, dataset=dataset, num_items=4, num_users=NUM_USERS,
    )
    r = m.fit(
        dataset,
        batch_size=-1,
        learning_rate=0.01,
        num_epochs=num_epochs,
        model_optimizer=optimizer,
        backend="lightning",
        print_summary=False,
    )
    n_params = int(r.coef_summary.shape[0])
    return float(r.train_ll), n_params, r

ll_a, k_a = float(result.train_ll), int(result.coef_summary.shape[0])

ll_b, k_b, result_b = fit_spec(
    "(itemsession_price|user) + (itemsession_feat|constant) + (intercept|item)",
    "B: HH price sensitivity",
)

ll_c, k_c, result_c = fit_spec(
    "(itemsession_price|user) + (itemsession_feat|user) + (intercept|item)",
    "C: full HH heterogeneity",
)

print(f"Spec A: LL={ll_a:.2f}, n_params={k_a}")
print(f"Spec B: LL={ll_b:.2f}, n_params={k_b}")
print(f"Spec C: LL={ll_c:.2f}, n_params={k_c}")


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Starting PyTorch Lightning training loop.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


...

/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
/Users/tianyudu/Development/torch-choice/.venv/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

/Users/tianyudu/Development/torch-choice/torch_choice/data/choice_dataset.py:286: UserWarning: The number of sessions is inferred from the number of unique sessions in the session_index tensor. This might lead to unexpected behaviors if some sessions never appeared in the session_index tensor. For a safer behavior, please provide the number of sessions explicitly by using the num_sessions keyword while initializing the ChoiceDataset class.
  warnings.warn(f"The number of sessions is inferred from the number of unique sessions in the session_index tensor. This might lead to unexpected behaviors if some sessions never appeared in the session_index tensor. For a safer behavior, please provide the number of sessions explicitly by using the num_sessions keyword while initializing the ChoiceDataset class.")


`Trainer.fit` stopped: `max_epochs=1000` reached.


Spec A: LL=-2656.89, n_params=5
Spec B: LL=-2085.90, n_params=104
Spec C: LL=-1980.28, n_params=203


### Compare via information criteria

Both AIC and BIC trade off goodness-of-fit (higher LL is better) against
complexity (more parameters is worse). BIC penalizes complexity more
aggressively, so it tends to favor parsimonious models — appropriate when we
believe the simpler spec is closer to the truth. AIC penalizes less and tends
to favor richer models when sample size is large.


In [9]:
specs = [
    ("A: pooled MNL",         ll_a, k_a),
    ("B: HH price sensitivity",  ll_b, k_b),
    ("C: full HH heterogeneity", ll_c, k_c),
]
table = pd.DataFrame([
    {
        "spec":   name,
        "LL":     ll,
        "n_params": k,
        "AIC":    2 * k - 2 * ll,
        "BIC":    k * math.log(N_OCC) - 2 * ll,
    }
    for name, ll, k in specs
])
table["ΔAIC vs A"] = table["AIC"] - table.iloc[0]["AIC"]
table["ΔBIC vs A"] = table["BIC"] - table.iloc[0]["BIC"]
table.set_index("spec")


,LL,n_params,AIC,BIC,ΔAIC vs A,ΔBIC vs A
spec,,,,,,
A: pooled MNL,-2656.887939,5,5323.775879,5352.716937,0.000000,0.000000
B: HH price sensitivity,-2085.898682,104,4379.797363,4981.771365,-943.978516,-370.945571
C: full HH heterogeneity,-1980.280518,203,4366.561035,5541.567981,-957.214844,188.851045


### Theoretical recommendation

The published literature on this dataset is unambiguous: **household
heterogeneity matters**, and the right model is one that lets price- and
feature-sensitivity vary across households.

- **Jain, Vilcassim & Chintagunta (1994)** reject pooled MNL on this exact
  dataset. Their random-coefficients logit shows that ignoring heterogeneity
  biases the average price coefficient toward zero (a classic "attenuation
  through aggregation" result).
- **Train (2009), *Discrete Choice Methods with Simulation*, Ch. 6** treats
  panel SP and RP data of this shape as the canonical motivating example for
  mixed logit, with random coefficients distributed normally or log-normally.
- **McFadden & Train (2000)** prove that mixed logit can approximate any
  random-utility model arbitrarily closely; the IIA assumption baked into
  pooled MNL is overly restrictive when tastes vary.

For Yogurt specifically, the recommended specification is therefore a **mixed
logit with random coefficients on price and feat**. `torch-choice` does not
yet ship a Monte-Carlo or quadrature-based mixed-logit estimator, so the
closest in-package approximations are Specs B and C, which estimate one
coefficient per household (fixed effects).

**Practical guidance for this notebook's reader:**

1. Use **Spec A** (pooled MNL) as a baseline, mainly to verify the workflow
   end-to-end and as a numerical reference (R/`mlogit` agreement, §4).
2. Use **Spec B** (per-household price coefficient) when the research
   question asks about *price elasticity heterogeneity* — e.g., does
   household X respond more to discounts than household Y? Spec B's
   identification is strongest where households face within-household price
   variation.
3. Use **Spec C** (per-household price *and* feature coefficient) when both
   marketing instruments are believed to elicit heterogeneous response. With
   the small sample (≈24 purchases per household), Spec C's per-household
   feat coefficient is identified primarily off households that experience
   features for multiple brands; expect noisier individual estimates.
4. The published JVC random-coefficients spec remains the gold standard;
   reach for it when `torch-choice` adds mixed-logit support, or use Apollo /
   Stata's `mixlogit` in the meantime.

**Caveat.** §5's alternative specs are *not* cross-validated against `mlogit`
(R doesn't natively fit per-user fixed-effects MNL with this many parameters
in the same form). The §4 verification covers Spec A only. Specs B and C are
internal-to-`torch-choice` model-fit comparisons.
